# Feature Engineering – NBA Player Longevity Prediction

**Feature Selection and Engineering Documentation:**
* **Target Definition**: The `target_5yrs` column is isolated to prevent data leakage during model training.
* **Noise Removal**: `Unnamed: 0` and `name` are removed as they are non-predictive identifiers.
* **Correlation & Redundancy**: Features with a correlation of 0.90 or higher (`fgm`, `fga`, `ftm`, `3pa`, `reb`) are dropped to resolve multicollinearity.
* **Composite Metrics**: New features (`pts_per_min`, `efficiency_rating`, `ast_tov_ratio`, `true_shooting_approx`) are engineered to capture normalized impact and per-minute efficiency.
* **Missing Values**: Infinite values are replaced with NaN, and missing numericals are handled via median imputation to avoid skew bias.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df_raw = pd.read_csv('0f484464-3cc8-4bfc-968b-b1a9fc4d4b1d.csv')

# Display initial dataset info
print("Initial Dataset Shape:", df_raw.shape)
display(df_raw.head())

# Define target variable
y = df_raw['target_5yrs'].copy()

Initial Dataset Shape: (1340, 22)


,Unnamed: 0,name,gp,min,pts,fgm,fga,fg,3p_made,3pa,...,fta,ft,oreb,dreb,reb,ast,stl,blk,tov,target_5yrs
0,0,Brandon Ingram,36,27.4,7.4,2.6,7.6,34.7,0.5,2.1,...,2.3,69.9,0.7,3.4,4.1,1.9,0.4,0.4,1.3,0
1,1,Andrew Harrison,35,26.9,7.2,2.0,6.7,29.6,0.7,2.8,...,3.4,76.5,0.5,2.0,2.4,3.7,1.1,0.5,1.6,0
2,2,JaKarr Sampson,74,15.3,5.2,2.0,4.7,42.2,0.4,1.7,...,1.3,67.0,0.5,1.7,2.2,1.0,0.5,0.3,1.0,0
3,3,Malik Sealy,58,11.6,5.7,2.3,5.5,42.6,0.1,0.5,...,1.3,68.9,1.0,0.9,1.9,0.8,0.6,0.1,1.0,1
4,4,Matt Geiger,48,11.5,4.5,1.6,3.0,52.4,0.0,0.1,...,1.9,67.4,1.0,1.5,2.5,0.3,0.3,0.4,0.8,1


In [3]:
# Drop non-predictive columns and the target variable from features to prevent leakage
# Note: 'Unnamed: 0' represents the raw index column from the CSV
DROP_COLS = ['Unnamed: 0', 'name', 'target_5yrs']
df = df_raw.drop(columns=DROP_COLS, errors='ignore').copy()
print("\nFeatures after dropping noise columns:", df.columns.tolist())


Features after dropping noise columns: ['gp', 'min', 'pts', 'fgm', 'fga', 'fg', '3p_made', '3pa', '3p', 'ftm', 'fta', 'ft', 'oreb', 'dreb', 'reb', 'ast', 'stl', 'blk', 'tov']


In [4]:
# Perform correlation analysis and drop highly redundant features (|r| >= 0.90)
# Dropping 'fgm' (corr w/ pts), 'fga' (corr w/ pts/fgm), 'ftm' (corr w/ fta), 
# '3pa' (corr w/ 3p_made), and 'reb' (derived from oreb + dreb)
DROP_REDUNDANT = ['fgm', 'fga', 'ftm', '3pa', 'reb']
df = df.drop(columns=DROP_REDUNDANT, errors='ignore')
print("\nFeatures after dropping redundant columns:", df.columns.tolist())


Features after dropping redundant columns: ['gp', 'min', 'pts', 'fg', '3p_made', '3p', 'fta', 'ft', 'oreb', 'dreb', 'ast', 'stl', 'blk', 'tov']


In [5]:
# Engineer composite features

# 1. Points Per Minute
df['pts_per_min'] = df['pts'] / (df['min'] + 1e-9)

# 2. Efficiency Rating (PER-style metric)
df['efficiency_rating'] = (df['pts'] + df['oreb'] + df['dreb'] + df['ast'] + df['stl'] + df['blk'] - df['tov']) / (df['min'] + 1e-9)

# 3. Assist-to-Turnover Ratio
df['ast_tov_ratio'] = df['ast'] / (df['tov'] + 0.001)

# 4. True Shooting Approximation
fg_pct = df['fg'] / 100
fga_est = np.where(fg_pct > 0, (df['pts'] - df['3p_made'] * 0.5) / (fg_pct + 1e-9), 0)
fga_est = np.clip(fga_est, 0.5, None)
df['true_shooting_approx'] = (df['pts'] / (2 * (fga_est + 0.44 * df['fta'] + 1e-9))).clip(0, 1)

print("\nEngineered new composite metrics successfully.")


Engineered new composite metrics successfully.


In [6]:
# Handle missing values and edge cases
# Replace any resulting infinite values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill NaNs using median imputation to prevent skewed averages from biasing data
cols_with_nulls = df.columns[df.isnull().any()].tolist()
if cols_with_nulls:
    df[cols_with_nulls] = df[cols_with_nulls].fillna(df[cols_with_nulls].median())

# Format final ML-ready dataset
df_final = df.copy()
df_final['target_5yrs'] = y.values

print("\nFinal ML-Ready Dataset Info:")
display(df_final.info())
display(df_final.head())


Final ML-Ready Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 1340 entries, 0 to 1339
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   gp                    1340 non-null   int64  
 1   min                   1340 non-null   float64
 2   pts                   1340 non-null   float64
 3   fg                    1340 non-null   float64
 4   3p_made               1340 non-null   float64
 5   3p                    1340 non-null   float64
 6   fta                   1340 non-null   float64
 7   ft                    1340 non-null   float64
 8   oreb                  1340 non-null   float64
 9   dreb                  1340 non-null   float64
 10  ast                   1340 non-null   float64
 11  stl                   1340 non-null   float64
 12  blk                   1340 non-null   float64
 13  tov                   1340 non-null   float64
 14  pts_per_min           1340 non-null   float64
 15  ef

None

,gp,min,pts,fg,3p_made,3p,fta,ft,oreb,dreb,ast,stl,blk,tov,pts_per_min,efficiency_rating,ast_tov_ratio,true_shooting_approx,target_5yrs
0,36,27.4,7.4,34.7,0.5,25.0,2.3,69.9,0.7,3.4,1.9,0.4,0.4,1.3,0.270073,0.470803,1.460415,0.171160,0
1,35,26.9,7.2,29.6,0.7,23.5,3.4,76.5,0.5,2.0,3.7,1.1,0.5,1.6,0.267658,0.498141,2.311056,0.146116,0
2,74,15.3,5.2,42.2,0.4,24.4,1.3,67.0,0.5,1.7,1.0,0.5,0.3,1.0,0.339869,0.535948,0.999001,0.209334,0
3,58,11.6,5.7,42.6,0.1,22.6,1.3,68.9,1.0,0.9,0.8,0.6,0.1,1.0,0.491379,0.698276,0.799201,0.206001,1
4,48,11.5,4.5,52.4,0.0,0.0,1.9,67.4,1.0,1.5,0.3,0.3,0.4,0.8,0.391304,0.626087,0.374532,0.238758,1
